# 01 — Exploratory Data Analysis

Goal: understand the dataset before modeling. We want to answer:
- How severe is the class imbalance?
- Are there missing values?
- What do the feature distributions look like?
- Are any features strongly correlated with fraud?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Consistent plot style throughout the notebook
sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load the Data

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Check for missing values — a critical first check before any modeling
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'None — dataset is complete')

## 2. Class Imbalance

This is the central challenge of this dataset. We expect ~0.17% fraud.

In [ ]:
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print('Class distribution:')
print(f'  Legitimate (0): {class_counts[0]:,} ({class_pct[0]:.2f}%)')
print(f'  Fraud (1):      {class_counts[1]:,} ({class_pct[1]:.2f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
sns.countplot(x='Class', data=df, ax=axes[0], palette=['steelblue', 'crimson'])
axes[0].set_title('Transaction Count by Class')
axes[0].set_xticklabels(['Legitimate', 'Fraud'])

# Pie chart shows the imbalance more intuitively
axes[1].pie(
    class_counts,
    labels=['Legitimate', 'Fraud'],
    autopct='%1.3f%%',
    colors=['steelblue', 'crimson'],
    startangle=90
)
axes[1].set_title('Class Distribution')

plt.tight_layout()
plt.show()

## 3. Amount and Time Distributions

`Amount` and `Time` are the only raw (non-PCA) features. We check if fraud transactions show different patterns.

In [ ]:
df.describe()

In [ ]:
# Amount distribution by class
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, label, color in zip(axes, ['Legitimate', 'Fraud'], ['steelblue', 'crimson']):
    class_val = 0 if label == 'Legitimate' else 1
    subset = df[df['Class'] == class_val]['Amount']
    ax.hist(subset, bins=50, color=color, alpha=0.7, edgecolor='white')
    ax.set_title(f'Transaction Amount — {label}')
    ax.set_xlabel('Amount ($)')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Legitimate — mean: ${df[df['Class']==0]['Amount'].mean():.2f}, max: ${df[df['Class']==0]['Amount'].max():.2f}")
print(f"Fraud      — mean: ${df[df['Class']==1]['Amount'].mean():.2f}, max: ${df[df['Class']==1]['Amount'].max():.2f}")

In [ ]:
# Time distribution — are fraud transactions clustered at certain times?
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, label, color in zip(axes, ['Legitimate', 'Fraud'], ['steelblue', 'crimson']):
    class_val = 0 if label == 'Legitimate' else 1
    subset = df[df['Class'] == class_val]['Time']
    ax.hist(subset, bins=50, color=color, alpha=0.7, edgecolor='white')
    ax.set_title(f'Transaction Time — {label}')
    ax.set_xlabel('Time (seconds from first transaction)')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

## 4. PCA Feature Distributions (V1–V28)

We can't interpret what these features represent, but features where the fraud and legitimate distributions diverge significantly will be the most predictive.

In [ ]:
v_features = [f'V{i}' for i in range(1, 29)]

fig, axes = plt.subplots(7, 4, figsize=(20, 28))
axes = axes.flatten()

for i, feature in enumerate(v_features):
    legitimate = df[df['Class'] == 0][feature]
    fraud = df[df['Class'] == 1][feature]
    axes[i].hist(legitimate, bins=50, alpha=0.5, color='steelblue', label='Legitimate', density=True)
    axes[i].hist(fraud, bins=50, alpha=0.5, color='crimson', label='Fraud', density=True)
    axes[i].set_title(feature)
    axes[i].legend(fontsize=7)

plt.suptitle('PCA Feature Distributions: Legitimate vs Fraud', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlation with Fraud Label

Which features have the strongest linear correlation with `Class`? High absolute correlation = more predictive signal.

In [ ]:
correlations = df.corr()['Class'].drop('Class').sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 8))
colors = ['crimson' if x > 0 else 'steelblue' for x in correlations]
correlations.plot(kind='barh', color=colors)
plt.title('Feature Correlation with Fraud Label')
plt.xlabel('Pearson Correlation')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print('Top 10 most correlated features with fraud:')
print(correlations.head(10))

## 6. Key Takeaways

Fill this in after running the notebook:

- **Class imbalance:** __% fraud — SMOTE resampling required
- **Missing values:** none
- **Amount:** fraud transactions tend to be ___ (smaller/larger/similar)
- **Time:** fraud ___ (clustered / evenly distributed)
- **Most predictive features:** V___, V___, V___ (based on correlation plot)
- **Feature engineering needed:** scale `Amount`, consider dropping or binning `Time`